In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_legal_action_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "LegalActionID String, SourceSystem String, PortfolioKey String, PortfolioName String, IngestionTimestamp timestamp,"
        #     "BatchId String, LegalActionKey String, LegalActionCode String, ExposureKey String, PartyKey String,"
        #     "CDI String, AgreementKey String,CaseCode string, CaseCodePrimary string, LegalActionCreatedDate timestamp, LegalActionCreatedBy String,"
        #     "LegalActionDate timestamp, LegalActionTypeCode String,LegalActionType String, LegalActionStatusCode String,"
        #     "LegalActionStatus String, LegalActionStepKey String, LegalActionStepTypeCode String,"
        #     "LegalActionStepType String, LegalActionStepStartDate timestamp,LegalActionStepEndDate timestamp,"
        #     "LegalTypeCode string,LegalType String, CompletionResultCode String, CompletionResultDate timestamp, CompletionResult String,"
        #     "LawyerPartyKey String,LawyerPartyName string, BankAccount string, Comments String, BusinessDate date"
        # )
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.LegalAction"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = ["PortfolioName", "Prefix", "unit_code_profits", "Arotron_dec_flag"] # Legal Actions QC Decommission to Arotron - notebooks run

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )
        portfolio_arotron_dec = portfolio.where(col("Arotron_dec_flag") == 0)
        portfolio_list = [row["PortfolioName"] for row in portfolio_arotron_dec.select("PortfolioName").collect()]


        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table_nj(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "arotron",
                portfolio_list
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== arxLegAct ==================

        arxLegAct_extra_filters = None

        if arxLegAct_extra_filters is not None:
            arxLegAct_extra_filters.append(date_filter)
            arxLegAct_final_filters = " AND ".join(arxLegAct_extra_filters)
        else:
            arxLegAct_final_filters = date_filter

        arxLegAct_selected_fields = [
            "PorfCode",
            "LegActID",
            "custcdi",
            "ContNum",
            "LegActDate",
            "LegActTypeID",
            "LegActStatusId",
            "LegActStepId",
            "LegActStepTypeId",
            "LegActStpStartDT",
            "LegActStpEndDT",
            "LegalType",
            "CompletionCode",
            "CompletionDate",
            "CompletionResult",
            "businessdate",
        ]

        arxLegAct = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_arxLegAct",
            arxLegAct_final_filters,
            arxLegAct_selected_fields,
        )
        # ================== ArxMData ==================

        ArxMData_extra_filters = None

        if ArxMData_extra_filters is not None:
            ArxMData_extra_filters.append(date_filter)
            ArxMData_final_filters = " AND ".join(ArxMData_extra_filters)
        else:
            ArxMData_final_filters = date_filter

        ArxMData_selected_fields = [
            "LstItemDescr",
            "LstItemRefCode",
        ]

        ArxMData = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_arxmdata",
            ArxMData_final_filters,
            ArxMData_selected_fields,
        )

        ArxMData_ActType = ArxMData.withColumnRenamed(
            "LstItemDescr", "LstItemDescr_ActType"
        ).withColumnRenamed("LstItemRefCode", "LstItemRefCode_ActType")

        ArxMData_StepType = ArxMData.withColumnRenamed(
            "LstItemDescr", "LstItemDescr_StepType"
        ).withColumnRenamed("LstItemRefCode", "LstItemRefCode_StepType")

        ArxMData_Type = ArxMData.withColumnRenamed(
            "LstItemDescr", "LstItemDescr_Type"
        ).withColumnRenamed("LstItemRefCode", "LstItemRefCode_Type")

        # ================== LegActBankAmount ==================

        LegActBankAmount_extra_filters = None

        if LegActBankAmount_extra_filters is not None:
            LegActBankAmount_extra_filters.append(date_filter)
            LegActBankAmount_final_filters = " AND ".join(
                LegActBankAmount_extra_filters
            )
        else:
            LegActBankAmount_final_filters = date_filter

        LegActBankAmount_selected_fields = [
            "BankAmount",
            "LegActID",
        ]

        LegActBankAmount = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_LegActBankAmount",
            LegActBankAmount_final_filters,
            LegActBankAmount_selected_fields,
        )


        # ================== Joins ==================
        target_table_arotron_primary_key_fields = [
            arxLegAct.LegActID,
            arxLegAct.LegActStepId,
            lit("arotron"),
        ]

        target_table_arotron_key_fields = [
            arxLegAct.LegActID,
            arxLegAct.LegActStepId,
        ]

        target_table_arotron = (
            arxLegAct.join(
                ArxMData_ActType,
                (
                    arxLegAct["LegActTypeID"]
                    == ArxMData_ActType["LstItemRefCode_ActType"]
                ),
                "left",
            )
            .join(
                ArxMData_StepType,
                (
                    arxLegAct["LegActStepTypeID"]
                    == ArxMData_StepType["LstItemRefCode_StepType"]
                ),
                "left",
            )
            .join(
                ArxMData_Type,
                (arxLegAct["LegalType"] == ArxMData_Type["LstItemRefCode_Type"]),
                "left",
            )
            .join(
                portfolio,
                substring(arxLegAct["CUSTCDI"], 1, 3) == portfolio["Prefix"],
                "left",
            )
            .join(
                LegActBankAmount,
                (LegActBankAmount["LegActID"] == arxLegAct["LegActID"]),
            )
        )

        target_table_arotron = target_table_arotron.withColumn(
            "LegalActionID",
            sha2(
                concat_ws(";", *target_table_arotron_primary_key_fields),
                256,
            ),
        ).withColumn("LegalActionKey", concat_ws(";", *target_table_arotron_key_fields)).distinct()

        # ================== Selection of Final Columns ==================

        result_df_arotron = target_table_arotron.select(
            target_table_arotron.LegalActionID,
            lit("arotron").cast("String").alias("SourceSystem"),
            coalesce(portfolio.unit_code_profits, lit("-9998")).cast("string").alias("PortfolioKey"),
            coalesce(portfolio.PortfolioName, lit("Core Wholesale")).alias(
                "PortfolioName"
            ),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            target_table_arotron.LegalActionKey,
            arxLegAct.LegActID.alias("LegalActionCode"),
            lit(None).cast("String").alias("ExposureKey"),
            lit(None).cast("String").alias("PartyKey"),
            arxLegAct.custcdi.cast("String").alias("CDI"),
            arxLegAct.ContNum.alias("AgreementKey"),
            lit(None).cast("String").alias("CaseCode"),
            lit("1").cast("String").alias("CaseCodePrimary"),
            lit(None).cast("timestamp").alias("LegalActionCreatedDate"),
            lit(None).cast("String").alias("LegalActionCreatedBy"),
            arxLegAct.LegActDate.cast("timestamp").alias("LegalActionDate"),
            arxLegAct.LegActTypeID.cast("String").alias("LegalActionTypeCode"),
            ArxMData_ActType.LstItemDescr_ActType.cast("String").alias(
                "LegalActionType"
            ),
            arxLegAct.LegActStatusId.cast("String").alias("LegalActionStatusCode"),
            arxLegAct.LegActStatusId.cast("String").alias("LegalActionStatus"),
            arxLegAct.LegActStepId.cast("String").alias("LegalActionStepKey"),
            arxLegAct.LegActStepTypeId.cast("String").alias("LegalActionStepTypeCode"),
            ArxMData_StepType.LstItemDescr_StepType.cast("String").alias(
                "LegalActionStepType"
            ),
            arxLegAct.LegActStpStartDT.cast("timestamp").alias(
                "LegalActionStepStartDate"
            ),
            arxLegAct.LegActStpEndDT.cast("timestamp").alias("LegalActionStepEndDate"),
            arxLegAct.LegalType.cast("String").alias("LegalTypeCode"),
            ArxMData_Type.LstItemDescr_Type.cast("String").alias("LegalType"),
            arxLegAct.CompletionCode.cast("String").alias("CompletionResultCode"),
            arxLegAct.CompletionDate.cast("timestamp").alias("CompletionResultDate"),
            arxLegAct.CompletionResult.cast("String").alias("CompletionResult"),
            lit(None).cast("String").alias("LawyerPartyKey"),
            lit(None).cast("String").alias("LawyerPartyName"),
            LegActBankAmount.BankAmount.cast("String").alias("BankAccount"),
            lit(None).cast("String").alias("Comments"),
            arxLegAct.businessdate.cast("date").alias("BusinessDate"),
            coalesce(portfolio.Arotron_dec_flag, lit("0")).alias("Arotron_dec_flag"), # Legal Actions QC Decommission to Arotron - notebooks run
        )

        # Legal Actions QC Decommission to Arotron - notebooks run
        # We keep only the portfolios for which the decommission of LAs has not yet happened
        result_df_arotron = result_df_arotron.filter("Arotron_dec_flag == 0")

        # Legal Actions QC Decommission to Arotron - notebooks run
        # We drop the column "Arotron_dec_flag" from the final dataframe
        result_df_arotron = result_df_arotron.drop("Arotron_dec_flag")

        # ================== Append ==================

        result_df_arotron.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_legal_action_arotron(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)
